# PATSTAT Reference — the citation edge list at application level

The twin of `PatentView/notebook/patent_reference.ipynb`: **the edges themselves**, one row per patent
citation, both ends resolved to a PATSTAT **application** (`appln_id`). `patstat_citation`,
`patstat_citation_trend`, `patstat_disruption` and `patstat_sb` all read this file instead of
re-scanning 508 M citation rows.

## Raw data
```
raw/tls212  citation      pat_publn_id (citing publication), cited_pat_publn_id | cited_appln_id, citn_origin, citn_replenished
raw/tls211  pat_publn     pat_publn_id -> appln_id, publn_date          (routing, both ends)
raw/tls201  appln         appln_id, appln_filing_year, ipr_type          (the universe, the time anchor)
```

## How a PATSTAT citation becomes an edge
A citation is made **by a publication** (an A1 search report, a B1 grant, ...) and points **at a
publication** (`cited_pat_publn_id`) or, when the office recorded the application itself, at an
application (`cited_appln_id`). Both are mapped to `appln_id` through tls211. Because one application
has several publications, the same citing application can cite the same target several times
(A1 and B1 both carry it; the A1 and the B1 of the target are both cited) -- exactly PatentView's
pre-grant / grant duplication. The file keeps every row; `patstat_citation` counts rows as `C` and
distinct citing applications as `uniqueC`.

## Conventions (switches in the first code cell)
- **Universe** on both ends: `ps.UNIVERSE_WHERE` -- patents of invention (`ipr_type = 'PI'`), real
  applications (`appln_id < 900 000 000`; PATSTAT's *artificial* applications hold cited documents it
  does not otherwise know), filing year 1900-2023. A citation to an artificial application is a citation
  to a document outside the index and is dropped, as PatentView drops citations to non-utility patents.
- **Time anchor = filing year** of the *application*, both ends (`age = citing_filing_year -
  cited_filing_year`). This is the PATSTAT / OECD convention; PatentView uses grant years because US
  utility patents all have one, most PATSTAT applications do not. `age` can be negative (a search
  report added years after filing cites a later-filed document); the file keeps such rows, the
  counting notebooks apply `ps.EDGE_WHERE` (`age >= 0 AND NOT replenished`).
- **Provenance** from `citn_origin` (`ps.ORIGIN_BUCKET`): `applicant` (APP), `examiner` (SEA, ISR,
  SUP, PRS, EXA, FOP, CH2), `other` (OPP, blank). **Third-party observations (115, TPO) are excluded**,
  as PatentView excludes `cited by third party`.
- **Replenished citations** (`citn_replenished <> 0`: copied onto a publication from another
  publication of the same family) are kept with `replenished = TRUE` so the file is complete, and
  filtered out downstream by default.
- **NPL** citations (`cited_npl_publn_id`) are not patent edges and are not here; their count per
  application is in `patstat_metadata.npl_ref_count`.

## Output
`PATSTAT/output/patstat_reference.parquet`

| column | meaning |
|---|---|
| `citing_id`, `cited_id` | `appln_id` of the citing / cited application (both in the universe) |
| `citing_publn_id`, `cited_publn_id` | the publications the citation was actually recorded on (`cited_publn_id` 0 when cited as an application) |
| `citing_year`, `cited_year` | filing years; `age = citing_year - cited_year` |
| `citing_publn_year` | year of the citing publication (an alternative, publication-based clock) |
| `citn_origin`, `bucket` | PATSTAT origin code and its examiner / applicant / other bucket |
| `replenished` | copied from a family member's publication |
| `resolved` | `publn` (via `cited_pat_publn_id`) or `appln` (via `cited_appln_id`) |

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_reference.parquet')
ps.preflight('patstat_reference')

UNIVERSE_BOTH_ENDS = True        # both citing and cited application must satisfy ps.UNIVERSE_WHERE
EXCLUDE_THIRD_PARTY = True       # drop citn_origin in ('115', 'TPO'), as PatentView drops 'cited by third party'
DROP_SELF = True                 # an application citing itself (via another of its publications)
con = ps.connect()
print('universe:', ps.UNIVERSE_WHERE)

## 1. Universe and publication routing

In [ ]:
%%time
# 1. The universe with its filing year, and the publication -> application routing table.
con.execute(f"""CREATE OR REPLACE TABLE uni AS
  SELECT appln_id, appln_filing_year AS fy FROM {ps.raw('tls201')} WHERE {ps.UNIVERSE_WHERE}""")
con.execute(f"""CREATE OR REPLACE TABLE pub AS
  SELECT pat_publn_id, appln_id, year(publn_date) AS py FROM {ps.raw('tls211')} WHERE pat_publn_id <> 0""")
n_uni = con.execute('SELECT count(*) FROM uni').fetchone()[0]
n_pub = con.execute('SELECT count(*) FROM pub').fetchone()[0]
print(f'universe: {n_uni:,} applications | publications: {n_pub:,}')

## 2. Resolve citations to applications

In [ ]:
%%time
# 2. Resolve every patent citation to (citing appln, cited appln). Audit the drops as we go.
tp = ps.third_party_sql('c.citn_origin')
tot = con.execute(f"""SELECT count(*) AS rows_total,
    count(*) FILTER (WHERE c.cited_pat_publn_id <> 0 OR c.cited_appln_id <> 0) AS patent_citations,
    count(*) FILTER (WHERE c.cited_pat_publn_id = 0 AND c.cited_appln_id = 0) AS npl_or_empty,
    count(*) FILTER (WHERE {tp}) AS third_party,
    count(*) FILTER (WHERE c.citn_replenished <> 0) AS replenished
  FROM {ps.raw('tls212')} c""").fetchdf()
display(tot)
con.execute(f"""CREATE OR REPLACE TABLE edges AS
  SELECT c.pat_publn_id AS citing_publn_id, pc.appln_id AS citing_id, pc.py AS citing_publn_year,
         COALESCE(pp.appln_id, NULLIF(c.cited_appln_id, 0)) AS cited_id,
         c.cited_pat_publn_id AS cited_publn_id,
         trim(c.citn_origin) AS citn_origin, {ps.bucket_sql('c.citn_origin')} AS bucket,
         c.citn_replenished <> 0 AS replenished,
         CASE WHEN pp.appln_id IS NOT NULL THEN 'publn' ELSE 'appln' END AS resolved
  FROM {ps.raw('tls212')} c
  JOIN pub pc ON c.pat_publn_id = pc.pat_publn_id
  LEFT JOIN pub pp ON c.cited_pat_publn_id <> 0 AND c.cited_pat_publn_id = pp.pat_publn_id
  WHERE (c.cited_pat_publn_id <> 0 OR c.cited_appln_id <> 0)
    {'AND NOT ' + tp if EXCLUDE_THIRD_PARTY else ''}""")
n_res = con.execute('SELECT count(*), count(cited_id) FROM edges').fetchone()
print(f'patent citations routed to a citing application: {n_res[0]:,}; with a resolvable cited application: {n_res[1]:,}')

## 3. Universe on both ends, filing years, write

In [ ]:
%%time
# 3. Restrict both ends to the universe, attach filing years, and write.
con.execute(f"""CREATE OR REPLACE TABLE ref AS
  SELECT e.citing_id, e.cited_id, e.citing_publn_id, e.cited_publn_id,
         uc.fy AS citing_year, ud.fy AS cited_year, (uc.fy - ud.fy)::INTEGER AS age, e.citing_publn_year,
         e.citn_origin, e.bucket, e.replenished, e.resolved
  FROM edges e
  JOIN uni uc ON e.citing_id = uc.appln_id
  JOIN uni ud ON e.cited_id = ud.appln_id
  {'WHERE e.citing_id <> e.cited_id' if DROP_SELF else ''}""")
con.execute(f"COPY ref TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)")
st = con.execute(f"""SELECT count(*) AS edges, count(DISTINCT citing_id) AS citing_apps, count(DISTINCT cited_id) AS cited_apps,
    round(100.0 * count(*) FILTER (WHERE age < 0) / count(*), 2) AS pct_age_negative,
    round(100.0 * count(*) FILTER (WHERE replenished) / count(*), 2) AS pct_replenished,
    round(100.0 * count(*) FILTER (WHERE resolved = 'appln') / count(*), 2) AS pct_resolved_via_appln
  FROM ref""").fetchdf()
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB)'); display(st)
print('by origin:')
display(con.execute("""SELECT citn_origin, bucket, count(*) AS n, round(100.0*count(*)/sum(count(*)) OVER (), 2) AS pct
  FROM ref GROUP BY 1, 2 ORDER BY n DESC""").fetchdf())
print('dropped because an end is outside the universe (artificial / non-PI / undated):',
      f'{n_res[1] - int(st.edges[0]):,}')
display(con.execute('SELECT * FROM ref LIMIT 8').fetchdf())
con.close()